In [55]:
import os
import pandas as pd
import numpy as np

In [56]:
os.chdir("C:/Users/ajayk/Desktop/quant_momentum")

In [57]:
import pickle

In [58]:
with open("all_stocks_last_6_years.pkl","rb") as file:
    df = pickle.load(file)

In [6]:
df['RELIANCE'].tail()

,symbol,open,high,low,close,volume
datetime,,,,,,
2025-08-28 20:45:00,NSE:RELIANCE,1381.1,1403.5,1350.0,1357.2,18758842.0
2025-08-31 20:45:00,NSE:RELIANCE,1356.0,1363.2,1340.6,1353.9,11232238.0
2025-09-01 20:45:00,NSE:RELIANCE,1354.8,1384.5,1354.5,1366.5,11517006.0
2025-09-02 20:45:00,NSE:RELIANCE,1369.7,1376.5,1360.5,1372.6,7847563.0
2025-09-03 20:45:00,NSE:RELIANCE,1371.8,1374.0,1362.6,1367.5,2239463.0


In [ ]:
data = pd.DataFrame()
for k, v in df.items():
    temp = v[['close']].rename(columns={'close': k})  # Correct rename syntax
    if data.empty:
        data = temp  # Directly assign for the first iteration
    else:
        data = pd.merge(data, temp, left_index=True, right_index=True, how='outer')  # Merge on index


In [59]:
data.shape

(0, 6)

In [11]:
data.to_csv("close_last_6_years.csv")
print(data.shape)

(7520, 2149)


In [13]:
df = pd.read_csv('close_last_6_years.csv',index_col='datetime')
df.index = pd.to_datetime(df.index).date

In [14]:
df.ffill(axis=0,inplace=True)

In [15]:
df.to_csv("close_6_years_ffill.csv")

In [16]:
df_market_cap = pd.read_csv("market_cap.csv")
df_market_cap.head(5)

,Symbol,Series,LTP,%chng,Mkt Cap (₹ Crores),Volume (Lakhs),Value (₹ Crores)
0,ZOMATO,EQ,274.70,-4.78,265095.13,2714.73,7675.63
1,IGIL,EQ,468.45,-8.15,20244.52,641.51,3174.33
2,KOTAKBANK,EQ,1748.70,-0.76,347670.26,146.86,2562.49
3,RELIANCE,EQ,1206.00,-1.99,1631993.75,203.13,2469.95
4,M&M,EQ,2916.95,-3.24,362731.14,80.46,2350.98


In [17]:
symbol_list = list(df_market_cap['Symbol '])
symbol_list = [x.strip() for x in symbol_list]

In [18]:
df_symbol = df.columns
common_symbol = df_symbol.intersection(symbol_list)

In [19]:
df = df[common_symbol]


In [20]:
df.index = pd.to_datetime(df.index)

df = df.ffill(axis=0)
df.dropna(how='all',axis=0,inplace=True)
df.dropna(how='all',axis=1,inplace=True)

In [22]:
df_monthly = df.resample("ME").min()
df_monthly.dropna(how='all',axis=0,inplace=True)
df_monthly.dropna(how='all',axis=1,inplace=True)
df_monthly = df_monthly.shift(1)
df_monthly = df_monthly.iloc[1:]

In [24]:
df_returns = df.pct_change()
df_returns.head(5)

,20MICRONS,21STCENMGM,3IINFOLTD,3MINDIA,3PLAND,5PAISA,63MOONS,A2ZINFRA,AAATECH,AAKASH,...,TIRUPATIFL,TRIDENT,VALIANTORG,VERANDA,CHEMFAB,INDIANHUME,MAYURUNIQ,PRIMESECU,TDPOWERSYS,WOCKPHARMA
1995-03-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-03-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-03-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-03-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-05-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
print(df_returns.shape)

(7520, 2004)


In [26]:
df_returns_adj = pd.DataFrame(data = np.where(df_returns<-0.3,0,df_returns),columns=df_returns.columns,index=df_returns.index)
df_returns_adj = pd.DataFrame(data = np.where(df_returns_adj > 0.3,0,df_returns_adj),columns=df_returns_adj.columns,index=df_returns_adj.index)
print(df_returns_adj.shape)


(7520, 2004)


In [39]:
def detect_momentum_crash(
    momentum,
    momentum_lookback=20,      # Days to compute momentum returns
    top_percentile=0.15,       # Top X% as high momentum
    bottom_percentile=0.15,    # Bottom X% as low momentum
    window=20,                 # Window size for rolling/expanding stats
    z_threshold=-1,            # Z-score threshold for crash signal
    mode="rolling"             # 'rolling' or 'from_start'
    ):
    """
    Detects momentum crashes using z-scores of high-minus-low momentum spread.
    
    Parameters:
        prices (pd.DataFrame): Asset prices indexed by date.
        momentum_lookback (int): Lookback period for momentum calculation.
        top_percentile (float): Fraction of assets in top momentum bucket.
        bottom_percentile (float): Fraction of assets in bottom momentum bucket.
        window (int): Lookback window for rolling mean/std (if mode='rolling').
        z_threshold (float): Z-score threshold for crash detection.
        mode (str): 'rolling' or 'from_start' for calculating mean/std.
    
    Returns:
        pd.Series: Boolean crash signal series indexed by date.
    """
    # Calculate momentum returns over the specified lookback
    #Replace Definition here if something else

    # Identify top and bottom momentum assets each day
    top = momentum.rank(axis=1, ascending=False) <= int(top_percentile * momentum.shape[1])
    bottom = momentum.rank(axis=1, ascending=False) >= int((1 - bottom_percentile) * momentum.shape[1])

    # Calculate average returns of top and bottom groups
    high_mom_returns = (momentum * top).mean(axis=1)
    low_mom_returns = (momentum * bottom).mean(axis=1)

    # Momentum spread = high momentum avg - low momentum avg
    momentum_spread = high_mom_returns - low_mom_returns

    # Calculate rolling or expanding statistics for z-score
    if mode == "rolling":
        mean_spread = momentum_spread.rolling(window).mean()
        std_spread = momentum_spread.rolling(window).std()
    elif mode == "from_start":
        mean_spread = momentum_spread.expanding().mean()
        std_spread = momentum_spread.expanding().std()
    else:
        raise ValueError("mode must be 'rolling' or 'from_start'")

    # Z-score calculation
    zscore = (momentum_spread - mean_spread) / std_spread

    # Crash signal where z-score falls below threshold
    crash_signal = zscore < z_threshold

    return crash_signal


In [31]:
crash_signal = detect_momentum_crash(df_returns_adj)

In [37]:
crash_signal_t = crash_signal[crash_signal==True]

In [38]:
crash_signal_t

1995-09-03    True
1995-09-05    True
1995-09-18    True
1995-10-12    True
1995-10-15    True
              ... 
2025-07-14    True
2025-07-16    True
2025-07-17    True
2025-07-24    True
2025-09-03    True
Length: 1064, dtype: bool

In [40]:

print(f"this is the shape of the df, {df.shape}")
print(f"this is the shape of the df monthly minimum value, {df_monthly.shape}")
print(f"this is the shape of the data before monthly calculation of df_returns_adj, {df_returns_adj.shape}")
df_m = df_returns_adj.resample('M').apply(lambda x: (1+x).prod() - 1 if x.notna().all() else np.nan)
print(f"this is the shape of the data after monthly calculation before nan removal, {df_m.shape}")
df_m.dropna(how='all',axis=1,inplace=True)
df_m.dropna(how='all',axis=0,inplace=True)
print(f"this is the shape of the data after monthly calculation, {df_m.shape}")

this is the shape of the df, (7520, 2004)
this is the shape of the df monthly minimum value, (365, 2004)
this is the shape of the data before monthly calculation of df_returns_adj, (7520, 2004)


C:\Users\ajayk\AppData\Local\Temp\ipykernel_18276\984772176.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_m = df_returns_adj.resample('M').apply(lambda x: (1+x).prod() - 1 if x.notna().all() else np.nan)


this is the shape of the data after monthly calculation before nan removal, (367, 2004)
this is the shape of the data after monthly calculation, (366, 2004)


In [41]:
crash_signal_m = detect_momentum_crash(df_m)

In [43]:
df_m.to_csv("df_m_last_6_yrs.csv")

In [44]:
df_m = pd.read_csv("df_m_last_6_yrs.csv",index_col="Unnamed: 0")
df_m.index = pd.to_datetime(df_m.index)
df_m.head(5)

,20MICRONS,21STCENMGM,3IINFOLTD,3MINDIA,3PLAND,5PAISA,63MOONS,A2ZINFRA,AAATECH,AAKASH,...,TIRUPATIFL,TRIDENT,VALIANTORG,VERANDA,CHEMFAB,INDIANHUME,MAYURUNIQ,PRIMESECU,TDPOWERSYS,WOCKPHARMA
1995-04-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1995-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.020000,NaN,NaN,NaN,NaN
1995-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
1995-08-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.029411,NaN,NaN,NaN,NaN


In [45]:
print(df_m.index[0],df_monthly.index[0])

1995-04-30 00:00:00 1995-05-31 00:00:00


In [46]:
print(df_m.index[-1])

2025-09-30 00:00:00


In [49]:
# import yfinance as yf

# # Define the stock ticker (e.g., 'AAPL' for Apple)
# ticker = "^NSEI"

# # Download historical data (default is 1 day of data)
# data = yf.download(ticker, start="1995-01-01", end="2025-09-02")



In [53]:
nsei_data_raw = pd.read_csv("nse_nifty_50_data.csv")
nsei_data_raw.set_index('Date',inplace=True)
nsei_data_raw.index = pd.to_datetime(nsei_data_raw.index).date
nsei_data_raw.sort_index(inplace=True)
nsei_data_raw.head(10)

,Price,Open,High,Low,Vol.,Change %
1995-11-06,988.92,"1,001.53","1,001.53",988.92,NaN,-1.11%
1995-11-07,978.22,987.17,987.17,977.05,NaN,-1.08%
1995-11-08,964.01,976.28,976.28,962.98,NaN,-1.45%
1995-11-09,953.07,960.32,960.32,952.13,NaN,-1.13%
1995-11-10,948.82,951.93,951.93,946,NaN,-0.45%
1995-11-13,917.26,949.29,949.29,916.48,NaN,-3.33%
1995-11-14,902.56,916.66,916.66,897.52,NaN,-1.60%
1995-11-15,913.21,901.33,913.54,901.33,NaN,1.18%
1995-11-16,904.08,915.96,915.96,900.83,NaN,-1.00%
1995-11-17,898.86,898.72,900.41,885.31,NaN,-0.58%


In [54]:
nsei_data = nsei_data_raw['Price']#data['Close']['^NSEI']
nsei_data_pct = nsei_data.pct_change()
nsei_data_m = nsei_data_pct.resample('ME').apply(lambda x: (1+x).prod() - 1 if x.notna().all() else np.nan)
nsei_data_m.rename("NSE",inplace=True)
nsei_data_m.head(2)


TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [42]:
df_m = df_m.merge(nsei_data_m,how='left',left_index=True,right_index=True)
df_m.head(2)

,21STCENMGM,3IINFOLTD,3MINDIA,3PLAND,5PAISA,63MOONS,A2ZINFRA,AAATECH,AADHARHFC,AAKASH,...,PARSVNATH,PHOENIXLTD,RUBYMILLS,SAKUMA,VAKRANGEE,WANBURY,KPIGREEN,MMP,STEELCAS,NSE
2000-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
risk_free = pd.read_csv("risk_free_rate.csv",index_col='Date')/100
risk_free.index = pd.to_datetime(risk_free.index,format="%Y-%m") + pd.offsets.MonthEnd(0)
risk_free.tail(10)

,RF
Date,
2024-03-31,0.005129
2024-04-30,0.005808
2024-05-31,0.005742
2024-06-30,0.005692
2024-07-31,0.005542
2024-08-31,0.005542
2024-09-30,0.005375
2024-10-31,0.005425
2024-11-30,0.005392


In [44]:
risk_free.loc['2025-01-31'] = 0.005598

In [45]:
df_m = df_m.merge(risk_free,how='left',left_index=True,right_index=True)
df_m.head(2)

,21STCENMGM,3IINFOLTD,3MINDIA,3PLAND,5PAISA,63MOONS,A2ZINFRA,AAATECH,AADHARHFC,AAKASH,...,PHOENIXLTD,RUBYMILLS,SAKUMA,VAKRANGEE,WANBURY,KPIGREEN,MMP,STEELCAS,NSE,RF
2000-05-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007066
2000-06-30 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007017


In [46]:
window = 60
beta_data = pd.DataFrame(index=df_m.index,columns=df_m.columns)
import statsmodels.api as sm
for i in range(window,len(df_m)):
    data = df_m.iloc[i-window:i,:]
    for col in data.columns:
        
        y = data[col] - data['RF']
        x = data['NSE'] - data['RF']
        
        
        if not x.isna().any() and not y.isna().any():
            x = sm.add_constant(x)
            # Fit the OLS model and store the beta value (coefficient for the market return)
            model = sm.OLS(y, x).fit()
            # Print the beta value for debugging
            print(model.params)
            # Store the beta value in beta_data
            beta_data.iloc[i, beta_data.columns.get_loc(col)] = model.params[0]
        else:
            # If there are NaNs, store NaN in beta_data
            beta_data.iloc[i, beta_data.columns.get_loc(col)] = np.nan
    

const   -0.006642
0       -0.001164
dtype: float64
const   -0.006642
0       -0.001164
dtype: float64
const   -0.006642
0       -0.001164
dtype: float64
const   -0.002894
0       -0.109067
dtype: float64
const    0.005120
0       -0.166552
dtype: float64
const   -0.006642
0       -0.001164
dtype: float64
const   -0.032675
0        1.589245
dtype: float64
const   -0.006642
0       -0.001164
dtype: float64
const    4.336809e-19
0        1.000000e+00
dtype: float64
const    0.0
0        0.0
dtype: float64
const   -0.006654
0       -0.001134
dtype: float64
const   -0.006654
0       -0.001134
dtype: float64
const   -0.006654
0       -0.001134
dtype: float64
const   -0.003808
0       -0.072612
dtype: float64
const   -0.003763
0        0.171740
dtype: float64
const   -0.006654
0       -0.001134
dtype: float64
const   -0.036713
0        1.737326
dtype: float64
const   -0.006654
0       -0.001134
dtype: float64
const    8.673617e-19
0        1.000000e+00
dtype: float64
const    0.0
0        0.0

In [48]:
beta_data.head(5)

,21STCENMGM,3IINFOLTD,3MINDIA,3PLAND,5PAISA,63MOONS,A2ZINFRA,AAATECH,AADHARHFC,AAKASH,...,PHOENIXLTD,RUBYMILLS,SAKUMA,VAKRANGEE,WANBURY,KPIGREEN,MMP,STEELCAS,NSE,RF
2000-05-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-06-30 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-07-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-08-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-09-30 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
beta_data.to_csv("beta_calc.csv")
beta_file = pd.read_csv("beta_calc.csv",index_col='Unnamed: 0')
beta_file.index = pd.to_datetime(beta_file.index)
print(beta_file.index[0])

2000-05-31 00:00:00


In [50]:
print(df_m.shape,beta_file.shape,df_monthly.shape)

(297, 2037) (297, 2037) (285, 2035)


In [56]:
df_m = df_m.iloc[:,:-2]
beta_file = beta_file.iloc[:,:-2]

In [68]:
def returns_portfolio_f2(data,beta_file,df_mnthly,shift_val=1,window=11,holding=1,decile=10,penny_price = 10):

    common_stocks = data.columns.intersection(beta_file.columns)

    data = data[common_stocks]
    beta_file = beta_file[common_stocks]
    df_mnthly = df_mnthly[common_stocks]
    
    index_common = data.index.intersection(df_mnthly.index)
    
    data = data.loc[index_common]
    beta_file = beta_file.loc[index_common]
    df_mnthly = df_mnthly.loc[index_common]

    def rolling_mean(returns, window):
            # Remove raw=True to allow pandas Series, which supports `notna()`
        return (1 + returns).rolling(window=window).apply(lambda x: np.prod(x) - 1 if x.notna().all() else np.nan)

    portfolio = rolling_mean(data.shift(shift_val),window=window)



    portfolio.dropna(how='all',axis=1,inplace=True)
    portfolio.dropna(how='all',axis=0,inplace=True)


    def assign_deciles(df,beta_file,df_mnthly,deciles):
        
        df_deciles = df.copy()
        

        
        beta_file = beta_file.loc[df_deciles.index]
        df_mnthly = df_mnthly.loc[df_deciles.index]
        
        beta_file = beta_file[df_deciles.columns]
        df_mnthly = df_mnthly[df_deciles.columns]
        
        for i, row in beta_file.iterrows():
            
            non_nan_values = row.dropna()
            
            if len(non_nan_values) > 0:
                # Find the top 10% of values
                top_10_percent = non_nan_values[non_nan_values >= non_nan_values.quantile(0.9)].index
                df_deciles.loc[i, top_10_percent] = np.nan  # Set the top 10% to NaN
        
        for i, row in df_mnthly.iterrows():
            non_nan_values = row.dropna()
            
            if len(non_nan_values)>  0:
                
                less_than_10 = non_nan_values[non_nan_values <= penny_price].index
                df_deciles.loc[i, less_than_10] = np.nan  # Set these to NaN
        
        for i, row in df_deciles.iterrows():
            non_nan_values = row.dropna()
            if len(non_nan_values) > 0:
                # Compute deciles for the remaining values
                top_10_percent = non_nan_values[non_nan_values >= non_nan_values.quantile(0.9)].index
                df_deciles.loc[i] = 0
                #decile_val = pd.qcut(non_nan_values, deciles, labels=False) + 1
                df_deciles.loc[i, top_10_percent] = decile
        
            
        return df_deciles

    df_deciles = assign_deciles(portfolio,beta_file,df_mnthly,decile)

    decile_portfolio = np.where(df_deciles==decile,1,0)
    decile_portfolio = pd.DataFrame(index=df_deciles.index,data=decile_portfolio,columns=df_deciles.columns)
    
    decile_portfolio.to_csv("january_31.csv")

    for i in range(0, len(df_deciles.index), holding):
        for j in range(1, holding):  # Start copying from the next column (i + 1, i + 2, ...)
            if i + j < len(df_deciles.index):  # Check to ensure we don't go out of bounds
                decile_portfolio.iloc[i+j, :] = decile_portfolio.iloc[i, :]
                

    ##decile_portfolio.to_csv("decile_portfolio_1923.csv")

    df_m2 = data[decile_portfolio.columns]
    df_m3 = df_m2.fillna(0)

    #df_m2.to_csv("Monthly Returns 1923.csv")
    returns_portfolio = pd.DataFrame(index=df_m3.index[window+shift_val:],columns=df_m3.columns,data=np.array(df_m3.iloc[window+shift_val:])*np.array(decile_portfolio.iloc[:-1]))
    returns_portfolio['sum'] = returns_portfolio.sum(axis=1)/np.sum(decile_portfolio,axis=1)

    s_sum = returns_portfolio['sum'].sum()
    sharpe = np.mean(returns_portfolio['sum'])*(12**0.5)/np.std(returns_portfolio['sum'])

    val_1 = val_1 = np.prod(1 + returns_portfolio['sum'])

    cagr = val_1**(12/len(returns_portfolio))-1

    print(f"shift = {shift_val} and window = {window} and holding = {holding} and sum is {s_sum} and sharpe is {sharpe} and cagr is {cagr}")
    
    return returns_portfolio

In [60]:
for hold in range(2,5):
    for window in range(6,12):
        returns_portfolio_f2(df_m,beta_file,df_monthly,shift_val=1,window=window,holding=hold,decile=10,penny_price = 10)

shift = 1 and window = 6 and holding = 2 and sum is 4.3372125214459025 and sharpe is 0.5500906871983611 and cagr is 0.13825581800982878
shift = 1 and window = 7 and holding = 2 and sum is 2.1811683245958697 and sharpe is 0.2953521786258566 and cagr is 0.0411203029133651
shift = 1 and window = 8 and holding = 2 and sum is 2.215781889746001 and sharpe is 0.32191278650580263 and cagr is 0.05083898233401185
shift = 1 and window = 9 and holding = 2 and sum is 2.1592587248477875 and sharpe is 0.32467470189540665 and cagr is 0.05237052813474641
shift = 1 and window = 10 and holding = 2 and sum is 2.7679640027842076 and sharpe is 0.4077007503477879 and cagr is 0.0793681170973588
shift = 1 and window = 11 and holding = 2 and sum is 1.5620110297436247 and sharpe is 0.2661864768047991 and cagr is 0.032397253100857615
shift = 1 and window = 6 and holding = 3 and sum is 3.3907741311551094 and sharpe is 0.4422251793249451 and cagr is 0.09495477024356358
shift = 1 and window = 7 and holding = 3 and s

In [61]:
for hold in range(2,5):
    for window in range(6,12):
        returns_portfolio_f2(data,beta_file,df_monthly,shift_val=1,window=window,holding=hold,decile=10,penny_price = 20)

# return returns_portfolio]

shift = 1 and window = 6 and holding = 2 and sum is 2.078378068331387 and sharpe is 2.248917873849012 and cagr is 0.5541055076033514
shift = 1 and window = 7 and holding = 2 and sum is 2.190305574845091 and sharpe is 2.396973762518303 and cagr is 0.6069116998809132
shift = 1 and window = 8 and holding = 2 and sum is 1.9577003036933007 and sharpe is 2.240791254218897 and cagr is 0.5403182172558028
shift = 1 and window = 9 and holding = 2 and sum is 2.1371626130725443 and sharpe is 2.461777928126685 and cagr is 0.6192051765211948
shift = 1 and window = 10 and holding = 2 and sum is 1.9409829779364207 and sharpe is 2.269245716904586 and cagr is 0.5610070185916656
shift = 1 and window = 11 and holding = 2 and sum is 2.02636071096971 and sharpe is 2.37800673505952 and cagr is 0.6075608191130188
shift = 1 and window = 6 and holding = 3 and sum is 2.1837168542065664 and sharpe is 2.252850523461575 and cagr is 0.5869289880018684
shift = 1 and window = 7 and holding = 3 and sum is 2.19653124300

In [66]:
def returns_portfolio_f3(data,beta_file,df_mnthly,shift_val=1,window=11,holding=1,decile=10,penny_price = 10,upper_price=10000):
    
    common_stocks = data.columns.intersection(beta_file.columns)

    data = data[common_stocks]
    beta_file = beta_file[common_stocks]
    df_mnthly = df_mnthly[common_stocks]
    
    index_common = data.index.intersection(df_mnthly.index)
    
    data = data.loc[index_common]
    beta_file = beta_file.loc[index_common]
    df_mnthly = df_mnthly.loc[index_common]

    def rolling_mean(returns, window):
            # Remove raw=True to allow pandas Series, which supports `notna()`
        return (1 + returns).rolling(window=window).apply(lambda x: np.prod(x) - 1 if x.notna().all() else np.nan)

    portfolio = rolling_mean(data.shift(shift_val),window=window)



    portfolio.dropna(how='all',axis=1,inplace=True)
    portfolio.dropna(how='all',axis=0,inplace=True)


    def assign_deciles(df,beta_file,df_mnthly,deciles):
        
        df_deciles = df.copy()
        

        
        beta_file = beta_file.loc[df_deciles.index]
        df_mnthly = df_mnthly.loc[df_deciles.index]
        
        beta_file = beta_file[df_deciles.columns]
        df_mnthly = df_mnthly[df_deciles.columns]
        
        for i, row in beta_file.iterrows():
            
            non_nan_values = row.dropna()
            
            if len(non_nan_values) > 0:
                # Find the top 10% of values
                top_10_percent = non_nan_values[non_nan_values >= non_nan_values.quantile(0.9)].index
                df_deciles.loc[i, top_10_percent] = np.nan  # Set the top 10% to NaN
        
        for i, row in df_mnthly.iterrows():
            non_nan_values = row.dropna()
            
            if len(non_nan_values)>  0:
                less_than_10 = non_nan_values[non_nan_values <= penny_price].index
                df_deciles.loc[i, less_than_10] = np.nan  # Set these to NaN
                greater_than_10k = non_nan_values[non_nan_values >= upper_price].index
                df_deciles.loc[i, greater_than_10k] = np.nan  # Set these to NaN
        
        for i, row in df_deciles.iterrows():
            non_nan_values = row.dropna()
            if len(non_nan_values) > 0:
                # Compute deciles for the remaining values
                top_10_percent = non_nan_values[non_nan_values >= non_nan_values.quantile(0.9)].index
                df_deciles.loc[i] = 0
                #decile_val = pd.qcut(non_nan_values, deciles, labels=False) + 1
                df_deciles.loc[i, top_10_percent] = decile
        
            
        return df_deciles

    df_deciles = assign_deciles(portfolio,beta_file,df_mnthly,decile)

    decile_portfolio = np.where(df_deciles==decile,1,0)
    decile_portfolio = pd.DataFrame(index=df_deciles.index,data=decile_portfolio,columns=df_deciles.columns)
    
    decile_portfolio.to_csv("january_31.csv")

    for i in range(0, len(df_deciles.index), holding):
        for j in range(1, holding):  # Start copying from the next column (i + 1, i + 2, ...)
            if i + j < len(df_deciles.index):  # Check to ensure we don't go out of bounds
                decile_portfolio.iloc[i+j, :] = decile_portfolio.iloc[i, :]
                

    ##decile_portfolio.to_csv("decile_portfolio_1923.csv")

    df_m2 = data[decile_portfolio.columns]
    df_m3 = df_m2.fillna(0)

    #df_m2.to_csv("Monthly Returns 1923.csv")
    returns_portfolio = pd.DataFrame(index=df_m3.index[window+1:],columns=df_m3.columns,data=np.array(df_m3.iloc[window+1:])*np.array(decile_portfolio.iloc[:-1]))
    returns_portfolio['sum'] = returns_portfolio.sum(axis=1)/np.sum(decile_portfolio,axis=1)

    s_sum = returns_portfolio['sum'].sum()
    sharpe = np.mean(returns_portfolio['sum'])*(12**0.5)/np.std(returns_portfolio['sum'])

    val_1 = val_1 = np.prod(1 + returns_portfolio['sum'])

    cagr = val_1**(12/len(returns_portfolio))-1

    print(f"shift = {shift_val} and window = {window} and holding = {holding} and sum is {s_sum} and sharpe is {sharpe} and cagr is {cagr}")
    
    return returns_portfolio

In [64]:
for hold in range(2,9):
    for window in range(1,15):
        returns_portfolio_f3(df_m,beta_file,df_monthly,shift_val=1,window=window,holding=hold,decile=10,penny_price = 50,upper_price=20000)

shift = 1 and window = 1 and holding = 2 and sum is 2.1662189670858685 and sharpe is 0.4091555851822364 and cagr is 0.06897067470835072
shift = 1 and window = 2 and holding = 2 and sum is 1.886548715429469 and sharpe is 0.3560917309450238 and cagr is 0.05656298750425903
shift = 1 and window = 3 and holding = 2 and sum is 1.43400284027335 and sharpe is 0.3611812135510499 and cagr is 0.046577951407755336
shift = 1 and window = 4 and holding = 2 and sum is 1.9342030807327815 and sharpe is 0.40582662488703036 and cagr is 0.062334166245589406
shift = 1 and window = 5 and holding = 2 and sum is 1.7313819710427016 and sharpe is 0.3225813392964679 and cagr is 0.04888692011528595
shift = 1 and window = 6 and holding = 2 and sum is 2.477889294365375 and sharpe is 0.48198814950685226 and cagr is 0.08604204573778396
shift = 1 and window = 7 and holding = 2 and sum is 1.811846430067828 and sharpe is 0.43469969165179656 and cagr is 0.06249485850224956
shift = 1 and window = 8 and holding = 2 and sum

In [65]:
for hold in range(2,5):
    for window in range(6,12):
        returns_portfolio_f2(data,beta_file,df_monthly,shift_val=1,window=window,holding=hold,decile=10,penny_price = 50)

# return returns_portfolio]

shift = 1 and window = 6 and holding = 2 and sum is 2.062546344893429 and sharpe is 2.345925557656792 and cagr is 0.551802670796715
shift = 1 and window = 7 and holding = 2 and sum is 2.061279479295748 and sharpe is 2.3758527568383343 and cagr is 0.5646620533571474
shift = 1 and window = 8 and holding = 2 and sum is 1.9113973140063962 and sharpe is 2.2711942583133142 and cagr is 0.526419502820654
shift = 1 and window = 9 and holding = 2 and sum is 2.052581625061664 and sharpe is 2.4881627783579736 and cagr is 0.5910406117986924
shift = 1 and window = 10 and holding = 2 and sum is 1.9181165707895969 and sharpe is 2.31186627771245 and cagr is 0.5545087130171382
shift = 1 and window = 11 and holding = 2 and sum is 1.9407682571598377 and sharpe is 2.3717950367677205 and cagr is 0.5774529125737453
shift = 1 and window = 6 and holding = 3 and sum is 2.0146710031020225 and sharpe is 2.1860738632384002 and cagr is 0.5327986877788682
shift = 1 and window = 7 and holding = 3 and sum is 2.1262902

In [69]:
for hold in range(3,4):
    for window in range(8,9):
        returns_portfolio_f2(data,beta_file,df_monthly,shift_val=1,window=window,holding=hold,decile=10,penny_price = 50)

# return returns_portfolio]

shift = 1 and window = 8 and holding = 3 and sum is 2.133501542634088 and sharpe is 2.463641712483916 and cagr is 0.6037594649918054
